# Intro to PySpark (optional — needs pyspark + Java)

## What you'll practice

Now the real thing — small, because the point of this lab is Python fluency. Every exercise here is a DataFrame version of something you already wrote by hand: explicit schemas (Level 12.2),
cleaning + latest-version dedup (Level 8), aggregation (Level 9), and partitioned overwrite writes (Level 8.6). Run `df.explain()` after each and look for `PartitionFilters`, `PushedFilters`,
`Exchange` (shuffle) and `BroadcastHashJoin`. Tests skip automatically if `pyspark` isn't installed.

> Function and class names in **Implement** are what the tests look for. Run all tests for this notebook with `pytest tests/pyspark_prep/test_12_04_intro_pyspark.py -v` from the lab root. Untouched exercises show as *skipped*.
>
> Solutions: `solutions/12_pyspark_preparation/04_intro_pyspark.ipynb`. Don't open it until you've tried.

### Setup — run this first (sample data and imports, not a solution)

In [ ]:
# Optional section: needs `pip install pyspark` and Java 17+ (see SETUP.md). Everything else in the lab works without it.
import os
os.environ.setdefault("PYSPARK_PYTHON", "python")
from pyspark.sql import SparkSession, functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = (SparkSession.builder.master("local[2]").appName("python-hands-on-lab")
         .config("spark.sql.shuffle.partitions", "4").config("spark.ui.enabled", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print(spark.version)

## Exercise 1 — 🟢 Easy · Read CSV with an explicit schema

**Implement:** `read_orders(spark, path)`

Read `datasets/csv/orders.csv` with an explicit `StructType` (all ten columns; `quantity` IntegerType, `unit_price` DoubleType, the rest StringType), `header=True`, and `mode="PERMISSIVE"`
so malformed numbers become null instead of failing. Return the DataFrame (lazy — nothing runs until an action).

**Input:** `spark, path`

**Constraints:** Standard library only (the last notebook uses the PySpark DataFrame API).

**Example:** the tests are executable examples — read `test_read_orders` in `tests/pyspark_prep/test_12_04_intro_pyspark.py`.

**Check:** `pytest tests/pyspark_prep/test_12_04_intro_pyspark.py -k test_read_orders`

**YOUR CODE ↓**

## Exercise 2 — 🟡 Medium · Clean and keep the latest version with a window

**Implement:** `clean_orders(df)`

Using `pyspark.sql.functions as F`: upper-case/trim `status`, lower-case `channel`, parse `order_ts`/`updated_at` with `F.to_timestamp(col, "yyyy-MM-dd'T'HH:mm:ss'Z'")`, keep rows with
`quantity >= 1`, non-null `unit_price >= 0`, non-null `customer_id` and parsed `order_ts`, add `amount = round(quantity * unit_price, 2)`, then keep the latest `updated_at` per `order_id` using
`row_number()` over a `Window.partitionBy("order_id").orderBy(F.col("updated_at").desc())`. Drop the helper column.

**Input:** `df`

**Constraints:** Standard library only (the last notebook uses the PySpark DataFrame API).

**Example:** the tests are executable examples — read `test_clean_orders` in `tests/pyspark_prep/test_12_04_intro_pyspark.py`.

**Check:** `pytest tests/pyspark_prep/test_12_04_intro_pyspark.py -k test_clean_orders`

**YOUR CODE ↓**

## Exercise 3 — 🟠 Hard · Aggregate daily revenue

**Implement:** `daily_revenue(df)`

From cleaned orders: exclude CANCELLED, group by `order_date` (= `F.to_date('order_ts')`) and `country`, aggregate `orders` (`countDistinct` order_id), `units` (sum quantity), `revenue` (sum amount, rounded 2). Order by `order_date`, `country`. Look at `explain()` and find the partial and final `HashAggregate`.

**Input:** `df`

**Constraints:** Standard library only (the last notebook uses the PySpark DataFrame API).

**Example:** the tests are executable examples — read `test_daily_revenue` in `tests/pyspark_prep/test_12_04_intro_pyspark.py`.

**Check:** `pytest tests/pyspark_prep/test_12_04_intro_pyspark.py -k test_daily_revenue`

**YOUR CODE ↓**

## Exercise 4 — 🔴 Very Hard · Partitioned, idempotent writes

**Implement:** `write_daily(df, path) -> None`

Write cleaned orders as Parquet partitioned by `order_date` with **dynamic partition overwrite** (`spark.sql.sources.partitionOverwriteMode=dynamic` set on the session conf, `mode("overwrite")`):
rewriting a day's data replaces only that day's partition, leaving others untouched — the Spark version of Level 8's `write_partition`. Then read back with a filter on `order_date` and check
`explain()` shows `PartitionFilters`.

**Input:** `df, path`

**Expected output:** `None`

**Constraints:** Standard library only (the last notebook uses the PySpark DataFrame API).

**Example:** the tests are executable examples — read `test_write_daily` in `tests/pyspark_prep/test_12_04_intro_pyspark.py`.

**Check:** `pytest tests/pyspark_prep/test_12_04_intro_pyspark.py -k test_write_daily`

**YOUR CODE ↓**